# CS103: Design and Analysis of Algorithms - Lab Assignment 11
**Sardar Vallabhbhai National Institute of Technology (SVNIT), Surat**  
**Department of Computer Science & Engineering**  
*M.Tech. (Data Science) I Year*

---

This notebook provides complete, clear, and self-contained Python solutions to all five graph algorithm problems given in **Assignment 11**:
1. **Assignment 1**: Design a Minimum-Cost One-Way Delivery Network Using Minimum Arborescence (Chu-Liu / Edmonds' Algorithm)
2. **Assignment 2**: Emergency Route Finder Using Dijkstra’s Algorithm
3. **Assignment 3**: Currency Exchange Arbitrage Detection Using Bellman–Ford
4. **Assignment 4**: Project Scheduling Using a Directed Acyclic Graph (DAG)
5. **Assignment 5**: Campus Navigation Using Breadth-First Search (BFS)

*Note: All implementations use standard Python libraries (`math`, `heapq`, `collections`) without requiring heavy external dependencies.*


---
## Assignment 1: Minimum-Cost One-Way Delivery Network Using Minimum Arborescence

### Problem Statement:
A logistics company has one central warehouse and several regional distribution centers. Possible one-way delivery routes have different costs. Design a minimum-cost directed distribution structure in which every regional center is reachable from the central warehouse and each non-root center has exactly one incoming route in the selected structure.

### Algorithm & Mathematical Formulation:
- **Graph Model**: Directed graph $G = (V, E)$ with edge weights $w(u, v)$ representing delivery route costs.
- **Root $r$**: Central warehouse.
- **Goal**: Find a directed spanning tree (arborescence) rooted at $r$ of minimum total weight such that every distribution center $v \in V \setminus \{r\}$ has in-degree exactly $1$ and can be reached from $r$.
- **Method**: **Chu-Liu / Edmonds' Algorithm**:
  1. For each non-root vertex $v$, select the incoming edge with minimum cost.
  2. If the selected edges form no directed cycles, they constitute the optimal minimum spanning arborescence.
  3. If directed cycles exist, contract each cycle into a pseudo-node, adjust incoming edge weights to account for cycle edge replacement: $w'(u, v_C) = w(u, v) - w(\text{in}(v))$, solve recursively, and expand the contracted cycles.


In [1]:
# Assignment 1: Minimum Arborescence (Chu-Liu / Edmonds' Algorithm)

def find_minimum_arborescence(nodes, root, edges):
    """
    Computes the minimum-cost directed spanning tree (arborescence) rooted at 'root'.
    
    Parameters:
      nodes : list of node identifiers (e.g. ['Warehouse', 'DC1', 'DC2', ...])
      root  : identifier of the central root node
      edges : list of tuples (u, v, cost) representing directed edge u -> v with weight 'cost'
      
    Returns:
      (total_cost, selected_edges) where selected_edges is the list of (u, v, cost) in the arborescence.
    """
    node_to_idx = {name: i for i, name in enumerate(nodes)}
    idx_to_node = {i: name for i, name in enumerate(nodes)}
    r_idx = node_to_idx[root]
    raw_edges = [(node_to_idx[u], node_to_idx[v], cost, i) for i, (u, v, cost) in enumerate(edges)]
    
    def edmonds(n, r, edge_list):
        # 1. Find cheapest incoming edge for each vertex v != r
        min_in = {}
        for u, v, w, eid in edge_list:
            if u != v and v != r:
                if v not in min_in or w < min_in[v][0]:
                    min_in[v] = (w, u, eid)
                    
        for v in range(n):
            if v != r and v not in min_in:
                return None, [] # Node unreachable from root
                
        # 2. Cycle detection in chosen minimum incoming edges
        comp = [-1] * n
        vis = [-1] * n
        num_cycles = 0
        cycle_edges = {}
        
        for i in range(n):
            if i == r:
                continue
            curr = i
            path = []
            while curr != r and curr != -1 and vis[curr] == -1:
                vis[curr] = i
                path.append(curr)
                curr = min_in[curr][1] if curr in min_in else -1
                
            if curr != -1 and curr != r and vis[curr] == i:
                cycle = []
                c = path.pop()
                cycle.append(c)
                while c != curr:
                    c = path.pop()
                    cycle.append(c)
                for node in cycle:
                    comp[node] = num_cycles
                    cycle_edges[node] = min_in[node]
                num_cycles += 1
                
        # Base case: acyclic selection -> minimum arborescence found!
        if num_cycles == 0:
            sel_eids = [min_in[v][2] for v in range(n) if v != r]
            cost = sum(min_in[v][0] for v in range(n) if v != r)
            return cost, sel_eids
            
        # 3. Contract cycles into pseudo-vertices
        comp_count = num_cycles
        for v in range(n):
            if comp[v] == -1:
                comp[v] = comp_count
                comp_count += 1
                
        contracted_edges = []
        for u, v, w, eid in edge_list:
            cu, cv = comp[u], comp[v]
            if cu != cv:
                if cv < num_cycles: # Edge entering a contracted cycle
                    contracted_edges.append((cu, cv, w - min_in[v][0], eid))
                else:
                    contracted_edges.append((cu, cv, w, eid))
                    
        cost, rec_eids = edmonds(comp_count, comp[r], contracted_edges)
        if cost is None:
            return None, []
            
        # 4. Expand contracted cycle edges
        cycle_incoming = {}
        for eid in rec_eids:
            for u, v, w, orig_eid in edge_list:
                if orig_eid == eid:
                    cv = comp[v]
                    if cv < num_cycles:
                        cycle_incoming[cv] = v
                    break
                    
        final_eids = set(rec_eids)
        for node, (cw, cu, ceid) in cycle_edges.items():
            cid = comp[node]
            if cid not in cycle_incoming or node != cycle_incoming[cid]:
                final_eids.add(ceid)
                
        return cost, list(final_eids)
        
    cost, sel_ids = edmonds(len(nodes), r_idx, raw_edges)
    if cost is None:
        return None, []
        
    selected_edges = [edges[i] for i in sel_ids]
    total_cost = sum(cost for u, v, cost in selected_edges)
    return total_cost, selected_edges

# Demonstration with Delivery Logistics Network:
nodes = ['Central Warehouse', 'DC North', 'DC South', 'DC East', 'DC West']
root = 'Central Warehouse'

delivery_routes = [
    ('Central Warehouse', 'DC North', 10),
    ('Central Warehouse', 'DC South', 15),
    ('DC North', 'DC East', 12),
    ('DC South', 'DC East', 8),
    ('DC East', 'DC West', 7),
    ('DC West', 'DC North', 9),
    ('Central Warehouse', 'DC West', 20),
    ('DC North', 'DC South', 5)
]

total_cost, arborescence_edges = find_minimum_arborescence(nodes, root, delivery_routes)

print("=" * 60)
print("ASSIGNMENT 1: MINIMUM ARBORESCENCE DELIVERY NETWORK")
print("=" * 60)
print(f"Central Warehouse (Root): {root}")
print(f"Regional Distribution Centers: {nodes[1:]}")
print(f"\nMinimum Total Network Delivery Cost: {total_cost}")
print("\nSelected One-Way Delivery Routes (Arborescence Edges):")
for u, v, cost in sorted(arborescence_edges, key=lambda x: (x[0], x[1])):
    print(f"  {u:20} ---> {v:15} | Route Cost: {cost}")


ASSIGNMENT 1: MINIMUM ARBORESCENCE DELIVERY NETWORK
Central Warehouse (Root): Central Warehouse
Regional Distribution Centers: ['DC North', 'DC South', 'DC East', 'DC West']

Minimum Total Network Delivery Cost: 30

Selected One-Way Delivery Routes (Arborescence Edges):
  Central Warehouse    ---> DC North        | Route Cost: 10
  DC East              ---> DC West         | Route Cost: 7
  DC North             ---> DC South        | Route Cost: 5
  DC South             ---> DC East         | Route Cost: 8


---
## Assignment 2: Emergency Route Finder Using Dijkstra’s Algorithm

### Problem Statement:
An ambulance service needs to find the shortest travel-time route from its current location to a hospital. The city is represented as a weighted graph, where intersections are vertices and roads are edges weighted by travel time. Implement Dijkstra’s algorithm to determine the shortest route.

### Algorithm & Mathematical Formulation:
- **Graph Model**: Graph $G = (V, E)$ where vertices are city intersections and edge weights $w(u, v) \ge 0$ denote travel times in minutes.
- **Method**: **Dijkstra's Algorithm** with a priority queue (`heapq` min-heap):
  - Maintains `dist[v]`, the minimum known travel time from start to intersection $v$.
  - Repeatedly extracts the unvisited vertex $u$ with minimum distance and relaxes outgoing edges $(u, v)$:
    $$\text{if } \text{dist}[u] + w(u, v) < \text{dist}[v] \implies \text{dist}[v] = \text{dist}[u] + w(u, v)$$
  - Tracks predecessor pointers to reconstruct the exact optimal path from the ambulance location to the hospital.


In [2]:
# Assignment 2: Emergency Route Finder (Dijkstra's Algorithm)
import heapq

def find_emergency_route(graph, start, destination):
    """
    Computes the shortest travel-time route and path from start to destination.
    
    Parameters:
      graph       : dict mapping node -> list of (neighbor, travel_time)
      start       : ambulance current location
      destination : hospital location
      
    Returns:
      (min_travel_time, path_list)
    """
    # Priority Queue stores: (travel_time, current_node, path)
    pq = [(0, start, [start])]
    visited = {}
    
    while pq:
        curr_time, u, path = heapq.heappop(pq)
        
        if u in visited and visited[u] <= curr_time:
            continue
        visited[u] = curr_time
        
        # Target reached
        if u == destination:
            return curr_time, path
            
        for v, travel_time in graph.get(u, []):
            if v not in visited or curr_time + travel_time < visited[v]:
                heapq.heappush(pq, (curr_time + travel_time, v, path + [v]))
                
    return None, []

# Demonstration with City Road Network:
city_intersections = {
    'Ambulance Station': [('Intersection A', 4), ('Intersection B', 2)],
    'Intersection A':    [('Intersection B', 1), ('Intersection C', 5), ('City Hospital', 14)],
    'Intersection B':    [('Intersection A', 1), ('Intersection C', 3), ('Intersection D', 8)],
    'Intersection C':    [('Intersection D', 2), ('City Hospital', 6)],
    'Intersection D':    [('City Hospital', 3)],
    'City Hospital':     []
}

start_loc = 'Ambulance Station'
hospital_loc = 'City Hospital'

travel_time, route = find_emergency_route(city_intersections, start_loc, hospital_loc)

print("=" * 60)
print("ASSIGNMENT 2: EMERGENCY AMBULANCE ROUTE FINDER (DIJKSTRA)")
print("=" * 60)
print(f"Origin:      {start_loc}")
print(f"Destination: {hospital_loc}")
print(f"\nMinimum Travel Time: {travel_time} minutes")
print(f"Optimal Emergency Route:\n  {' -> '.join(route)}")


ASSIGNMENT 2: EMERGENCY AMBULANCE ROUTE FINDER (DIJKSTRA)
Origin:      Ambulance Station
Destination: City Hospital

Minimum Travel Time: 10 minutes
Optimal Emergency Route:
  Ambulance Station -> Intersection B -> Intersection C -> Intersection D -> City Hospital


---
## Assignment 3: Currency Exchange Arbitrage Detection Using Bellman–Ford

### Problem Statement:
A financial platform receives exchange rates between different currencies. It must detect whether a sequence of currency exchanges can yield a profit by converting money through a cycle. Model the exchange network using logarithmic edge weights and apply Bellman–Ford to detect negative-weight cycles.

### Algorithm & Mathematical Formulation:
- **Arbitrage Condition**:
  Converting along a cycle of currencies $c_1 \to c_2 \to \dots \to c_k \to c_1$ with exchange rates $R(c_i, c_{i+1})$ yields a risk-free profit if:
  $$\prod_{i=1}^{k} R(c_i, c_{i+1}) > 1.0$$
- **Logarithmic Transformation**:
  Taking natural logarithm on both sides:
  $$\ln\left(\prod_{i=1}^{k} R(c_i, c_{i+1})\right) > 0 \iff \sum_{i=1}^{k} \ln(R(c_i, c_{i+1})) > 0 \iff \sum_{i=1}^{k} -\ln(R(c_i, c_{i+1})) < 0$$
  Therefore, setting edge weight $w(u, v) = -\ln(R(u, v))$ transforms the product maximization into finding a **negative-weight cycle**.
- **Bellman-Ford Algorithm**:
  - Distances initialized to $0.0$ for all currencies (detects cycles in any connected component).
  - Relax all edges $|V| - 1$ times.
  - On the $|V|$-th pass, if any edge relaxes, a negative cycle exists.
  - Backtrack $|V|$ predecessor steps to isolate a vertex in the cycle, then extract the complete cycle sequence.


In [3]:
# Assignment 3: Currency Exchange Arbitrage Detection (Bellman-Ford)
import math

def detect_currency_arbitrage(currencies, rates):
    """
    Detects whether an arbitrage cycle exists using Bellman-Ford on log-transformed weights.
    
    Parameters:
      currencies : list of currency symbols (e.g. ['USD', 'EUR', 'GBP', 'JPY'])
      rates      : list of (u, v, exchange_rate)
      
    Returns:
      (cycle_path, profit_multiplier) if arbitrage exists, else (None, 1.0)
    """
    # Transform exchange rate to negative logarithm: w = -ln(rate)
    edges = []
    rate_map = {}
    for u, v, r in rates:
        w = -math.log(r)
        edges.append((u, v, w))
        rate_map[(u, v)] = r
        
    n = len(currencies)
    dist = {c: 0.0 for c in currencies}
    parent = {c: None for c in currencies}
    
    # Relax all edges |V| - 1 times
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v] - 1e-9:
                dist[v] = dist[u] + w
                parent[v] = u
                
    # |V|-th pass: Check for relaxation indicating a negative cycle
    cycle_vertex = None
    for u, v, w in edges:
        if dist[u] + w < dist[v] - 1e-9:
            parent[v] = u
            cycle_vertex = v
            break
            
    if cycle_vertex is None:
        return None, 1.0 # No arbitrage opportunity found
        
    # Step back |V| times to ensure we enter the cycle
    curr = cycle_vertex
    for _ in range(n):
        curr = parent[curr]
        
    # Reconstruct the cycle
    cycle = [curr]
    p = parent[curr]
    while p != curr:
        cycle.append(p)
        p = parent[p]
    cycle.append(curr)
    cycle.reverse()
    
    # Compute overall multiplier across the cycle
    multiplier = 1.0
    for i in range(len(cycle) - 1):
        multiplier *= rate_map[(cycle[i], cycle[i+1])]
        
    return cycle, multiplier

# Demonstration Case 1: Market WITH Arbitrage Opportunity
market_currencies = ['USD', 'EUR', 'GBP', 'JPY']
# USD -> EUR (0.90) * EUR -> GBP (0.80) * GBP -> USD (1.42) = 1.0224 (> 1) -> 2.24% profit!
arbitrage_rates = [
    ('USD', 'EUR', 0.90),
    ('EUR', 'USD', 1.0 / 0.90),
    ('EUR', 'GBP', 0.80),
    ('GBP', 'EUR', 1.0 / 0.80),
    ('GBP', 'USD', 1.42),
    ('USD', 'GBP', 1.0 / 1.42),
    ('USD', 'JPY', 152.0),
    ('JPY', 'USD', 1.0 / 152.0)
]

cycle, multiplier = detect_currency_arbitrage(market_currencies, arbitrage_rates)

print("=" * 60)
print("ASSIGNMENT 3: CURRENCY ARBITRAGE DETECTION (BELLMAN-FORD)")
print("=" * 60)
print("Scenario 1: Market With Arbitrage Opportunity")
if cycle:
    profit_pct = (multiplier - 1.0) * 100
    print("  [+] Arbitrage Opportunity Detected!")
    print(f"  Cycle Route:       {' -> '.join(cycle)}")
    print(f"  Product of Rates:  {multiplier:.6f}")
    print(f"  Net Profit Return: +{profit_pct:.2f}% (Starting with 1,000 {cycle[0]} yields {1000 * multiplier:.2f} {cycle[0]})")
else:
    print("  [-] No Arbitrage Opportunity Detected.")

# Demonstration Case 2: Fair Efficient Market (No Arbitrage)
fair_rates = [
    ('USD', 'EUR', 0.92),
    ('EUR', 'USD', 1.0 / 0.92),
    ('EUR', 'GBP', 0.85),
    ('GBP', 'EUR', 1.0 / 0.85),
    ('GBP', 'USD', 1.0 / (0.92 * 0.85)),
    ('USD', 'GBP', 0.92 * 0.85),
]
no_cycle, no_mult = detect_currency_arbitrage(['USD', 'EUR', 'GBP'], fair_rates)
print("\nScenario 2: Efficient Market (Fair Exchange Rates)")
if no_cycle:
    print("  Arbitrage Detected:", ' -> '.join(no_cycle))
else:
    print("  [-] No Arbitrage Opportunity Detected (Market is efficient and balanced).")


ASSIGNMENT 3: CURRENCY ARBITRAGE DETECTION (BELLMAN-FORD)
Scenario 1: Market With Arbitrage Opportunity
  [+] Arbitrage Opportunity Detected!
  Cycle Route:       USD -> EUR -> GBP -> USD
  Product of Rates:  1.022400
  Net Profit Return: +2.24% (Starting with 1,000 USD yields 1022.40 USD)

Scenario 2: Efficient Market (Fair Exchange Rates)
  [-] No Arbitrage Opportunity Detected (Market is efficient and balanced).


---
## Assignment 4: Project Scheduling Using a Directed Acyclic Graph (DAG)

### Problem Statement:
A software development team has tasks that depend on the completion of other tasks. Represent the tasks and dependencies as a Directed Acyclic Graph (DAG). Determine a valid task execution order and calculate the earliest completion time of the project.

### Algorithm & Mathematical Formulation:
- **Graph Model**: Directed Acyclic Graph $G = (V, E)$, where each task $u \in V$ has duration $d(u)$, and directed edge $(u, v) \in E$ indicates task $u$ must finish before task $v$ can begin.
- **Topological Sorting**: Using **Kahn's Algorithm** (in-degree tracking and queue) to generate a valid sequence of tasks respecting all dependency constraints.
- **Earliest Completion Time & Critical Path**:
  - $\text{EST}[v] = \max_{(u, v) \in E}(\text{EFT}[u])$, with $\text{EST}[\text{start}] = 0$
  - $\text{EFT}[v] = \text{EST}[v] + d(v)$
  - Total Earliest Project Completion Time:
    $$T = \max_{v \in V}(\text{EFT}[v])$$
  - The **Critical Path** is the sequence of bottleneck tasks where any delay directly extends total project completion time.


In [4]:
# Assignment 4: Project Scheduling in DAG (Topological Sort & Longest Path)
from collections import deque

def schedule_project(tasks, dependencies):
    """
    Determines valid execution order and computes earliest completion time of the project.
    
    Parameters:
      tasks        : dict of {task_name: duration_in_days}
      dependencies : list of (u, v) indicating u must finish before v can start
      
    Returns:
      dict containing topological order, earliest start times, earliest finish times,
      total project completion time, and critical path.
    """
    adj = {t: [] for t in tasks}
    in_degree = {t: 0 for t in tasks}
    for u, v in dependencies:
        adj[u].append(v)
        in_degree[v] += 1
        
    # Kahn's Algorithm for Topological Sort
    queue = deque([t for t in tasks if in_degree[t] == 0])
    topo_order = []
    while queue:
        u = queue.popleft()
        topo_order.append(u)
        for v in adj[u]:
            in_degree[v] -= 1
            if in_degree[v] == 0:
                queue.append(v)
                
    if len(topo_order) < len(tasks):
        raise ValueError("Cycle detected! Project dependencies must form a Directed Acyclic Graph (DAG).")
        
    # Compute Earliest Start Time (EST) and Earliest Finish Time (EFT)
    est = {t: 0 for t in tasks}
    eft = {t: tasks[t] for t in tasks}
    predecessors = {t: None for t in tasks}
    
    for u in topo_order:
        eft[u] = est[u] + tasks[u]
        for v in adj[u]:
            if eft[u] > est[v]:
                est[v] = eft[u]
                predecessors[v] = u
                
    total_duration = max(eft.values())
    
    # Trace Critical Path (path leading to maximum completion time)
    last_task = max(tasks.keys(), key=lambda t: eft[t])
    critical_path = []
    curr = last_task
    while curr:
        critical_path.append(curr)
        curr = predecessors[curr]
    critical_path.reverse()
    
    return {
        'topological_order': topo_order,
        'earliest_start': est,
        'earliest_finish': eft,
        'total_completion_time': total_duration,
        'critical_path': critical_path
    }

# Demonstration with Software Development Project:
dev_tasks = {
    '1. Requirements Analysis': 3,
    '2. Architecture Design':   4,
    '3. Frontend Development':  6,
    '4. Backend Development':   8,
    '5. Database Setup':        3,
    '6. API Integration':       4,
    '7. Quality Testing (QA)':  5,
    '8. Production Deployment': 2
}

dev_dependencies = [
    ('1. Requirements Analysis', '2. Architecture Design'),
    ('2. Architecture Design',   '3. Frontend Development'),
    ('2. Architecture Design',   '5. Database Setup'),
    ('5. Database Setup',        '4. Backend Development'),
    ('3. Frontend Development',  '6. API Integration'),
    ('4. Backend Development',   '6. API Integration'),
    ('6. API Integration',       '7. Quality Testing (QA)'),
    ('7. Quality Testing (QA)',  '8. Production Deployment')
]

result = schedule_project(dev_tasks, dev_dependencies)

print("=" * 60)
print("ASSIGNMENT 4: PROJECT SCHEDULING USING DAG")
print("=" * 60)
print("Valid Execution Order (Topological Sequence):")
for step, task in enumerate(result['topological_order'], 1):
    print(f"  Step {step}: {task}")

print(f"\nEarliest Overall Project Completion Time: {result['total_completion_time']} days")
print(f"Critical Path (Bottleneck Path):\n  {' -> '.join(result['critical_path'])}")

print("\nTask Schedule Details (Days):")
print(f"  {'Task':<28} | {'Duration':<9} | {'EST':<5} | {'EFT':<5}")
print("  " + "-" * 54)
for t in result['topological_order']:
    print(f"  {t:<28} | {dev_tasks[t]:<9} | {result['earliest_start'][t]:<5} | {result['earliest_finish'][t]:<5}")


ASSIGNMENT 4: PROJECT SCHEDULING USING DAG
Valid Execution Order (Topological Sequence):
  Step 1: 1. Requirements Analysis
  Step 2: 2. Architecture Design
  Step 3: 3. Frontend Development
  Step 4: 5. Database Setup
  Step 5: 4. Backend Development
  Step 6: 6. API Integration
  Step 7: 7. Quality Testing (QA)
  Step 8: 8. Production Deployment

Earliest Overall Project Completion Time: 29 days
Critical Path (Bottleneck Path):
  1. Requirements Analysis -> 2. Architecture Design -> 5. Database Setup -> 4. Backend Development -> 6. API Integration -> 7. Quality Testing (QA) -> 8. Production Deployment

Task Schedule Details (Days):
  Task                         | Duration  | EST   | EFT  
  ------------------------------------------------------
  1. Requirements Analysis     | 3         | 0     | 3    
  2. Architecture Design       | 4         | 3     | 7    
  3. Frontend Development      | 6         | 7     | 13   
  5. Database Setup            | 3         | 7     | 10   
  4. B

---
## Assignment 5: Campus Navigation Using Breadth-First Search (BFS)

### Problem Statement:
A university wants a navigation system that finds the route with the fewest walkway segments between buildings. Represent buildings as vertices and walkways as unweighted edges. Implement BFS to find the minimum number of segments from a starting building to a destination.

### Algorithm & Mathematical Formulation:
- **Graph Model**: Unweighted graph $G = (V, E)$ where vertices represent campus buildings and edges represent direct walkway connections.
- **Method**: **Breadth-First Search (BFS)**:
  - BFS visits all neighbors at distance $k$ before any neighbor at distance $k+1$.
  - This property guarantees finding the path with the minimum number of edges (walkway segments) in $O(V + E)$ time.
  - Maintains `parent` mapping to reconstruct the shortest route sequence from start to destination.


In [5]:
# Assignment 5: Campus Navigation (Breadth-First Search)
from collections import deque

def campus_navigation_bfs(walkways, start_building, destination_building):
    """
    Finds the route with fewest walkway segments between start and destination buildings.
    
    Parameters:
      walkways             : dict mapping building -> list of directly connected buildings
      start_building       : starting building name
      destination_building : destination building name
      
    Returns:
      (num_segments, path_sequence)
    """
    if start_building == destination_building:
        return 0, [start_building]
        
    visited = {start_building}
    parent = {start_building: None}
    queue = deque([start_building])
    
    while queue:
        curr = queue.popleft()
        if curr == destination_building:
            break
            
        for neighbor in walkways.get(curr, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = curr
                queue.append(neighbor)
                
    if destination_building not in visited:
        return None, [] # Destination is unreachable
        
    # Reconstruct path
    path = []
    curr = destination_building
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()
    
    return len(path) - 1, path

# Demonstration with SVNIT Campus Map:
svnit_walkways = {
    'Main Gate':                  ['Administrative Building', 'Central Library', 'Student Canteen'],
    'Administrative Building':    ['Main Gate', 'Computer Engineering Dept', 'Electrical Engineering Dept'],
    'Central Library':            ['Main Gate', 'Computer Engineering Dept', 'Mechanical Engineering Dept'],
    'Student Canteen':            ['Main Gate', 'Bhabha Bhavan Hostel', 'Student Activity Center'],
    'Computer Engineering Dept':  ['Administrative Building', 'Central Library', 'Central Computer Center'],
    'Electrical Engineering Dept':['Administrative Building', 'Mechanical Engineering Dept'],
    'Mechanical Engineering Dept':['Central Library', 'Electrical Engineering Dept', 'Central Workshop'],
    'Bhabha Bhavan Hostel':       ['Student Canteen', 'Sports Complex'],
    'Student Activity Center':    ['Student Canteen', 'Sports Complex'],
    'Central Computer Center':    ['Computer Engineering Dept', 'Central Workshop'],
    'Central Workshop':           ['Mechanical Engineering Dept', 'Central Computer Center', 'Sports Complex'],
    'Sports Complex':             ['Bhabha Bhavan Hostel', 'Student Activity Center', 'Central Workshop']
}

start_bldg = 'Main Gate'
dest_bldg = 'Central Workshop'

segments, route = campus_navigation_bfs(svnit_walkways, start_bldg, dest_bldg)

print("=" * 60)
print("ASSIGNMENT 5: CAMPUS NAVIGATION (BFS)")
print("=" * 60)
print(f"Origin Building:      {start_bldg}")
print(f"Destination Building: {dest_bldg}")
print(f"\nMinimum Walkway Segments: {segments}")
print(f"Optimal Campus Route:\n  {' -> '.join(route)}")


ASSIGNMENT 5: CAMPUS NAVIGATION (BFS)
Origin Building:      Main Gate
Destination Building: Central Workshop

Minimum Walkway Segments: 3
Optimal Campus Route:
  Main Gate -> Central Library -> Mechanical Engineering Dept -> Central Workshop


---
## Summary of Solutions & Complexity Analysis

| Assignment | Problem Domain | Algorithm Applied | Time Complexity | Space Complexity | Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Assignment 1** | Delivery Network Design | Chu-Liu / Edmonds' Algorithm | $O(V \cdot E)$ | $O(V + E)$ | Verified |
| **Assignment 2** | Emergency Route Finder | Dijkstra's with Min-Heap | $O((V + E) \log V)$ | $O(V)$ | Verified |
| **Assignment 3** | Currency Arbitrage Detection | Bellman-Ford on $-\ln(R)$ | $O(V \cdot E)$ | $O(V)$ | Verified |
| **Assignment 4** | Project Scheduling | Topological Sort + DAG Longest Path | $O(V + E)$ | $O(V)$ | Verified |
| **Assignment 5** | Campus Walkway Navigation | Breadth-First Search (BFS) | $O(V + E)$ | $O(V)$ | Verified |

All problems solved using simple, self-contained Python standard library implementations.
